# Cookbook: Web IQ — Grounding Agents with Real-Time Web Search

Some questions can only be answered with information that changes by the hour: today's news, current prices, the latest release notes. **Web IQ** gives Foundry agents a built-in **web search tool** that retrieves fresh public web content before the model responds, and returns the answer with inline citations.

In this cookbook, you will:

- **Create a Foundry agent** with the `WebSearchTool` attached
- **Ask a time-sensitive question** and stream the agent's response
- **Inspect inline citations** linking back to the web pages used to ground the answer
- **Scope the search** to a specific location
- **Clean up** the agent when you're done

## Prerequisites

You need:

1. A **Microsoft Foundry** project with a basic or standard [Foundry Agent Service environment](https://learn.microsoft.com/azure/ai-foundry/agents/environment-setup).
2. A **model deployment** that supports the web search tool (e.g., `gpt-5-mini`). `gpt-4o-mini` and GPT-5 reasoning models are **not** supported.
3. The **Foundry User** role on the project to create and run agents.

Create a `.env` file in this directory with your endpoint values:

```
FOUNDRY_PROJECT_ENDPOINT=https://<your-ai-services>.services.ai.azure.com/api/projects/<your-project>
FOUNDRY_MODEL_DEPLOYMENT_NAME=gpt-5-mini
```

> **💡 No new infrastructure needed.** Unlike Foundry IQ, this cookbook doesn't require an Azure AI Search index, a knowledge source, or a knowledge base. If you already deployed a Foundry project for the Foundry IQ episodes, reuse it here.

## Setup

Install required packages and load environment variables. We authenticate with `DefaultAzureCredential` so no API keys are stored in code.

> **Before you start:** Run `az login` in a terminal to sign in to Azure. This is required for `DefaultAzureCredential` to work.

In [ ]:
%%capture
%pip install -U azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv(override=True)

# Service endpoints
PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT_NAME = os.environ.get("FOUNDRY_MODEL_DEPLOYMENT_NAME", "gpt-5-mini")

credential = DefaultAzureCredential()
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
openai = project.get_openai_client()

print(f"Project endpoint: {PROJECT_ENDPOINT}")
print(f"Model deployment: {MODEL_DEPLOYMENT_NAME}")

---

## Step 1 — Create an Agent with the Web Search Tool

The `WebSearchTool` is attached directly to a `PromptAgentDefinition` — no separate Bing resource or project connection is required for general web search. Optionally, `WebSearchApproximateLocation` biases results toward a region.

In [ ]:
from azure.ai.projects.models import (
    PromptAgentDefinition,
    WebSearchTool,
    WebSearchApproximateLocation,
)

agent = project.agents.create_version(
    agent_name="web-iq-episode1-agent",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions="You are a helpful assistant that can search the web for current information. Always cite your sources.",
        tools=[
            WebSearchTool(
                user_location=WebSearchApproximateLocation(
                    country="US", city="Seattle", region="Washington"
                ),
            )
        ],
    ),
    description="Episode 1 Web IQ agent grounded in live web search.",
)

print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

---

## Step 2 — Ask a Time-Sensitive Question

We send a query that requires current information and stream the response. `tool_choice=\"required\"` forces the model to use the web search tool rather than answer from its own training data.

In [ ]:
stream_response = openai.responses.create(
    stream=True,
    tool_choice="required",
    input="What is today's date, and what is a notable technology news headline from today?",
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)

citations = []
for event in stream_response:
    if event.type == "response.created":
        print(f"Response created with ID: {event.response.id}\n")
    elif event.type == "response.output_text.delta":
        print(event.delta, end="")
    elif event.type == "response.output_item.done":
        if event.item.type == "message":
            item = event.item
            if item.content and item.content[-1].type == "output_text":
                for annotation in item.content[-1].annotations:
                    if annotation.type == "url_citation":
                        citations.append(annotation.url)
    elif event.type == "response.completed":
        print("\n\n✅ Response completed")

---

## Step 3 — Inspect the Citations

Per the [Grounding with Bing use and display requirements](https://www.microsoft.com/en-us/bing/apis/grounding-legal#use-and-display-requirements), website citations must be displayed in the exact form provided. Here we print the URLs collected from the streamed response's `url_citation` annotations.

In [ ]:
print("📎 Citations returned with this response:\n")
for url in citations:
    print(f"- {url}")

if not citations:
    print("No citations were returned — try re-running with a more time-sensitive question.")

---

## Step 4 — Try a Few More Questions

Run the cell below with different prompts to see how the agent decides when to search the web versus answer directly.

In [ ]:
def ask(question: str) -> None:
    response = openai.responses.create(
        input=question,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    )
    print(f"❓ {question}\n")
    print(f"📝 {response.output_text}\n")

ask("What is the current weather in Seattle?")
ask("What are the latest headlines about the Microsoft IQ Series?")

---

## Clean Up

Delete the agent version created in this cookbook. If you want to keep experimenting, skip this cell.

In [ ]:
project.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print(f"✅ Agent '{agent.name}' version '{agent.version}' deleted")

---

## Conclusion

In this cookbook we walked through grounding a Foundry agent in the live public web:

1. **Web search tool** — We attached `WebSearchTool` directly to a prompt agent, with no Bing resource or knowledge base to provision.
2. **Grounded response** — We asked a time-sensitive question and streamed back an answer built from current web content.
3. **Citations** — We inspected the `url_citation` annotations that let end users trace every claim back to its source.
4. **Location scoping** — We saw how `WebSearchApproximateLocation` can bias results toward a specific region.

### Where to go next

- **Combine with Foundry IQ** — Give the same agent both the web search tool and a knowledge base MCP tool, so it can ground answers in your organizational knowledge *and* the live web.
- **Try domain-restricted search** — Use Grounding with Bing Custom Search to scope results to an allow-list of domains. See the [web search tool documentation](https://learn.microsoft.com/azure/ai-foundry/agents/how-to/tools/web-search).
- **Use a toolbox** — Centralize credential management and reuse the web search tool across agents with a [toolbox](https://learn.microsoft.com/azure/ai-foundry/agents/concepts/toolbox-overview).
- **Explore the IQ Series** — Head back to the [Web IQ overview](../../README.md) or continue with [Foundry IQ](../../../Foundry-IQ/) and [Work IQ](../../../Work-IQ/).